In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import box
from collections import defaultdict
from tqdm import tqdm

# ===================== User Inputs =====================
depth_tif_2021 = "../LCMAP_CU_2021_V13_LCPRI.tif"
shapefile = "../CONUS.shp"
edge_files = {
    "north": "../EdgeAge_north_2021.tif",
    "south": "../EdgeAge_south_2021.tif",
    "east":  "../EdgeAge_east_2021.tif",
    "west":  "../EdgeAge_west_2021.tif"
}

state_to_region_division = {
    'Connecticut': ('Northeast', 'New England'), 'Maine': ('Northeast', 'New England'),
    'Massachusetts': ('Northeast', 'New England'), 'New Hampshire': ('Northeast', 'New England'),
    'Rhode Island': ('Northeast', 'New England'), 'Vermont': ('Northeast', 'New England'),
    'New Jersey': ('Northeast', 'Middle Atlantic'), 'New York': ('Northeast', 'Middle Atlantic'),
    'Pennsylvania': ('Northeast', 'Middle Atlantic'),
    'Illinois': ('Midwest', 'East North Central'), 'Indiana': ('Midwest', 'East North Central'),
    'Michigan': ('Midwest', 'East North Central'), 'Ohio': ('Midwest', 'East North Central'),
    'Wisconsin': ('Midwest', 'East North Central'),
    'Iowa': ('Midwest', 'West North Central'), 'Kansas': ('Midwest', 'West North Central'),
    'Minnesota': ('Midwest', 'West North Central'), 'Missouri': ('Midwest', 'West North Central'),
    'Nebraska': ('Midwest', 'West North Central'), 'North Dakota': ('Midwest', 'West North Central'),
    'South Dakota': ('Midwest', 'West North Central'),
    'Delaware': ('South', 'South Atlantic'), 'District of Columbia': ('South', 'South Atlantic'),
    'Florida': ('South', 'South Atlantic'), 'Georgia': ('South', 'South Atlantic'),
    'Maryland': ('South', 'South Atlantic'), 'North Carolina': ('South', 'South Atlantic'),
    'South Carolina': ('South', 'South Atlantic'), 'Virginia': ('South', 'South Atlantic'),
    'West Virginia': ('South', 'South Atlantic'),
    'Alabama': ('South', 'East South Central'), 'Kentucky': ('South', 'East South Central'),
    'Mississippi': ('South', 'East South Central'), 'Tennessee': ('South', 'East South Central'),
    'Arkansas': ('South', 'West South Central'), 'Louisiana': ('South', 'West South Central'),
    'Oklahoma': ('South', 'West South Central'), 'Texas': ('South', 'West South Central'),
    'Arizona': ('West', 'Mountain'), 'Colorado': ('West', 'Mountain'), 'Idaho': ('West', 'Mountain'),
    'Montana': ('West', 'Mountain'), 'Nevada': ('West', 'Mountain'), 'New Mexico': ('West', 'Mountain'),
    'Utah': ('West', 'Mountain'), 'Wyoming': ('West', 'Mountain'),
    'Alaska': ('West', 'Pacific'), 'California': ('West', 'Pacific'),
    'Hawaii': ('West', 'Pacific'), 'Oregon': ('West', 'Pacific'), 'Washington': ('West', 'Pacific')
}

# ===================== Constants =====================
MIN_AGE = 1
MAX_AGE = 34          # known maximum edge age
AGE_RANGE = np.arange(MIN_AGE, MAX_AGE + 1)  # 1..34
PER_EDGE_LEN_M = 30.0

# ===================== Prepare Geometries =====================
with rasterio.open(depth_tif_2021) as sample_raster:
    raster_crs = sample_raster.crs
    bounds = sample_raster.bounds
    extent_geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)

conus_gdf = gpd.read_file(shapefile)
if conus_gdf.crs != raster_crs:
    conus_gdf = conus_gdf.to_crs(raster_crs)

# clip to raster extent to drop AK/HI when using CONUS rasters
conus_clipped = gpd.clip(conus_gdf, extent_geom)
conus_clipped = conus_clipped[["geometry", "NAME", "STUSPS"]]

# Add Region info and keep only states with a known region
conus_clipped["Region"] = conus_clipped["NAME"].map(lambda x: state_to_region_division.get(x, (None, None))[0])
conus_clipped = conus_clipped[~conus_clipped["Region"].isna()]

# Dissolve polygons into 4 regions
region_gdf = conus_clipped.dissolve(by="Region")
region_order = [r for r in ["South", "Midwest", "Northeast", "West"] if r in region_gdf.index]

# ===================== Helpers =====================
def get_region_age_counts(src, geom):
    """
    Return a length-(MAX_AGE) array of counts for ages 1..MAX_AGE within geom.
    Uses crop=True to read the smallest window. Zeros and <=0 are ignored.
    """
    # Choose a safe fill value; if nodata is None, we fill with 0
    fill_val = src.nodata if src.nodata is not None else 0

    # Crop to geometry for efficiency; fill outside with 'fill_val'
    out_img, _ = mask(src, [geom], crop=True, filled=True, nodata=fill_val)
    data = out_img[0]

    # Valid: >0 and <= MAX_AGE (ignore 0/negatives/out-of-range)
    valid = (data > 0) & (data <= MAX_AGE)
    if not np.any(valid):
        return np.zeros(MAX_AGE, dtype=np.int64)  # all zeros for ages 1..34

    vals = data[valid].astype(np.int32)
    # bincount across 0..MAX_AGE; slice 1..MAX_AGE
    bc = np.bincount(vals, minlength=MAX_AGE + 1)[MIN_AGE:MAX_AGE + 1]
    return bc.astype(np.int64)

def counts_to_mean_age(counts):
    """Length-weighted mean equals age-weighted mean since each pixel == 30 m."""
    total = counts.sum()
    if total == 0:
        return np.nan
    return float((AGE_RANGE * counts).sum() / total)

# ===================== Main Computation =====================
orientations = ["north", "south", "east", "west"]

# histograms per orientation per region (Series indexed by age 1..34)
hist_dfs_per_orient = {}
# pooled counts over orientations per region
pooled_counts_by_region = {r: np.zeros(MAX_AGE, dtype=np.int64) for r in region_order}
# mean rows to summarize
mean_rows = []

outer = tqdm(orientations, desc="Orientations", position=0)
for orient in outer:
    orient_hist_per_region = {}
    with rasterio.open(edge_files[orient]) as src:
        inner = tqdm(region_order, desc=f"{orient}→regions", position=1, leave=False)
        for region in inner:
            geom = region_gdf.loc[region, "geometry"]
            counts = get_region_age_counts(src, geom)  # length 34 counts for ages 1..34

            # Store histogram (per age) for this orientation×region
            orient_hist_per_region[region] = pd.Series(counts, index=AGE_RANGE)

            # Update pooled counts by region across orientations
            pooled_counts_by_region[region] += counts

            # Compute stats
            n_edges = int(counts.sum())
            length_m = int(n_edges * PER_EDGE_LEN_M)
            lw_mean = counts_to_mean_age(counts)

            mean_rows.append({
                "orientation": orient,
                "region": region,
                "n_edges": n_edges,
                "length_m": length_m,
                "mean_age_length_weighted": lw_mean
            })

    # finalize this orientation's histogram DataFrame
    hist_dfs_per_orient[orient] = pd.DataFrame(orient_hist_per_region)

# Pooled (ALL orientations) per region
pooled_hist_df = pd.DataFrame(
    {region: pd.Series(pooled_counts_by_region[region], index=AGE_RANGE) for region in region_order}
)

# Pooled means
for region in region_order:
    counts = pooled_counts_by_region[region]
    n_edges = int(counts.sum())
    length_m = int(n_edges * PER_EDGE_LEN_M)
    lw_mean = counts_to_mean_age(counts)
    mean_rows.append({
        "orientation": "ALL",
        "region": region,
        "n_edges": n_edges,
        "length_m": length_m,
        "mean_age_length_weighted": lw_mean
    })

# Final summary table
mean_df = (
    pd.DataFrame(mean_rows)
    .sort_values(["orientation", "region"])
    .reset_index(drop=True)
)

print(mean_df)

print(hist_dfs_per_orient["north"])        # ages as rows, regions as columns
print(hist_dfs_per_orient["south"])        # ages as rows, regions as columns
print(hist_dfs_per_orient["east"])        # ages as rows, regions as columns
print(hist_dfs_per_orient["west"])        # ages as rows, regions as columns
print(pooled_hist_df)                      # pooled over orientations


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== Inputs expected from your previous step =====
# mean_df columns: ['orientation','region','n_edges','length_m','mean_age_length_weighted']
# pooled_hist_df: index = ages (1..34), columns = ['South','Midwest','Northeast','West']
# If your column order differs, set region_order accordingly:
region_order = [r for r in ["South", "Midwest", "Northeast", "West"] if r in pooled_hist_df.columns]

# ===== Helper to convert counts to km for bar lengths =====
def counts_to_km(counts, meters_per_edge=30.0):
    return (counts * meters_per_edge) / 1000.0

# ===== Prepare data for panel (a) =====
mean_series = (
    mean_df[mean_df["orientation"]=="ALL"]
    .set_index("region")
    .loc[region_order, "mean_age_length_weighted"]
)

# ===== Colors (roughly matching your example) =====
colors = {
    "CONUS": "#7f7f7f",      # gray
    "South": "#1f77b4",      # blue
    "Midwest": "#d4d000",    # yellow-ish
    "Northeast": "#6f2dbd",  # purple-ish
    "West": "#d62728",       # red
}

# ===== Make figure =====
fig, axes = plt.subplots(2, 3, figsize=(14, 6), constrained_layout=True, dpi=600)
(ax_a, ax_b, ax_c), (ax_d, ax_e, ax_f) = axes

# ---------- (a) Mean years by region ----------
ax_a.bar(["CONUS"] + region_order,
         [ (mean_df[mean_df["orientation"]=="ALL"]["mean_age_length_weighted"]
             .mean()) ] + list(mean_series.values),
         color=[colors["CONUS"]] + [colors[r] for r in region_order],
         edgecolor="black", linewidth=0.6)
ax_a.set_ylabel("Mean Years Since Edge Created")
ax_a.set_title("(a) Mean Years Since Edge Created by Region",
               loc="left", fontweight="bold")
ax_a.set_ylim(0, max(35, np.nanmax(mean_series.values)*1.15))

# ---------- function to draw a horizontal histogram panel ----------
def plot_region_hist(ax, region, panel_letter):
    ages = pooled_hist_df.index.values  # 1..34
    counts = pooled_hist_df[region].values
    lengths_km = counts_to_km(counts)

    ax.barh(ages, lengths_km, height=0.8, color=colors[region], edgecolor="black", linewidth=0.3)
    ax.set_ylim(0, ages.max()+1)
    ax.set_xlabel("Total Edge Length (km)")
    ax.set_ylabel("Years Since Edge Created")
    ax.set_title(f"({panel_letter}) {region}", loc="left", fontweight="bold")

    yticks = np.arange(0, ages.max()+1, 5)
    yticks[0] = 1
    ax.set_yticks(yticks)



# ---------- (b)(c)(e)(f) per-region ----------
# map regions to subplots like your layout
plot_region_hist(ax_b, "Northeast", "b")
plot_region_hist(ax_c, "Midwest",   "c")
plot_region_hist(ax_e, "West",      "e")
plot_region_hist(ax_f, "South",     "f")

# ---------- (d) CONUS pooled ----------
ax_d.barh(pooled_hist_df.index.values, conus_km, height=0.8,
          color=colors["CONUS"], edgecolor="black", linewidth=0.3)
ax_d.set_ylim(0, pooled_hist_df.index.max()+1)
ax_d.set_xlabel("Total Edge Length (km)")
ax_d.set_ylabel("Years Since Edge Created")
ax_d.set_title("(d) CONUS", loc="left", fontweight="bold")

# ---------- tidy look ----------
for ax in [ax_b, ax_c, ax_d, ax_e, ax_f]:
    ax.grid(True, axis="x", linestyle=":", linewidth=0.6, alpha=0.6)
for ax in axes.flatten():
    ax.tick_params(labelsize=9)

plt.savefig(r"Figure_S2.png", dpi=600, bbox_inches='tight')
plt.show()